In [1]:
import os
from dotenv import load_dotenv
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
import fitz  # PyMuPDF
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.prompts import PromptTemplate
from operator import itemgetter


/Users/Shared/RAG-for-drug-pdf-files/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load environment variables from .env file
load_dotenv()

# Retrieve API key from .env file
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("MODEL")
PDF_FOLDER = os.getenv("PDF_FOLDER")
ONEDRIVE_PATH_PDF = os.getenv("ONEDRIVE_PATH_PDF")

# Initialize GPT model and embeddings from OpenAI
model = ChatOpenAI(openai_api_key=OPENAI_API_KEY, model=MODEL, temperature=0)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)


In [3]:
# Example GPT model invocation
response = model.invoke("Tell me a joke")
print(response)

content="Why couldn't the bicycle stand up by itself?\n\nBecause it was two tired!" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 11, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-85347642-5a79-4409-9477-473597a3348b-0' usage_metadata={'input_tokens': 11, 'output_tokens': 17, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [4]:
response

AIMessage(content="Why couldn't the bicycle stand up by itself?\n\nBecause it was two tired!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 11, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-85347642-5a79-4409-9477-473597a3348b-0', usage_metadata={'input_tokens': 11, 'output_tokens': 17, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [5]:
parser = StrOutputParser()

chain = model | parser 
chain.invoke("Tell me a joke")

'Why did the scarecrow win an award? Because he was outstanding in his field!'

In [6]:
# Check if the PDF folder is defined
if not PDF_FOLDER:
    raise ValueError("The path to the PDF folder is not defined in the .env file.")

In [8]:

# Variable to store text from all PDF files
all_text = ""

# Iterate over all PDF files in the folder
for filename in os.listdir(PDF_FOLDER):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(PDF_FOLDER, filename)
        
        try:
            # Open the PDF using PyMuPDF (fitz)
            doc = fitz.open(pdf_path)

            # Iterate over pages and collect text
            for page_num in range(doc.page_count):
                page = doc.load_page(page_num)
                all_text += page.get_text()  # Extract text from each page

        except Exception as e:
            print(f"Error processing file {filename}: {e}")

# After collecting all text, split it into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
chunks = splitter.split_text(all_text)

# Print the number of chunks
print(f"Number of chunks: {len(chunks)}")

# Display the first chunk
if chunks:
    print(f"Content of the first chunk:\n{chunks[167]}")


Number of chunks: 27512
Content of the first chunk:
HAE przejawia się w postaci przemijających napadów obrzęku podskórnego i (lub) 
podśluzówkowego, obejmującego górne drogi oddechowe, skórę i przewód pokarmowy. Napad 
zwykle trwa od 2 do 5 dni. 
 
Ikatybant jest selektywnym, kompetycyjnym antagonistą receptora bradykininy typu 2 (B2). Jest to 
7 
 
syntetyczny dekapeptyd o strukturze podobnej do bradykininy, lecz zawierający 5 aminokwasów 
niebiałkogennych. W przebiegu HAE bradykinina występująca w zwiększonym stężeniu jest 
kluczowym mediatorem w rozwoju objawów klinicznych. 
 
Działanie farmakodynamiczne 
 
U zdrowych, młodych osób ikatybant podawany w dawkach 0,8 mg/kg mc. przez 4 godziny, 
1,5 mg/kg mc./dobę lub 0,15 mg/kg mc./dobę przez 3 dni zapobiegał indukowanej bradykininą 
hipotonii, rozszerzeniu naczyń i odruchowej tachykardii. Wykazano, że ikatybant jest kompetycyjnym 
antagonistą w warunkach, gdy dawkę bradykininy w teście prowokacji zwiększono 4-krotnie. 
 
Skuteczność k

In [9]:
# Load or create embeddings
def load_or_create_embeddings(source_text_chunks, onedrive_path):
    faiss_index_path = os.path.join(onedrive_path, "index.faiss")

    if os.path.exists(faiss_index_path):
        print(f"Loading existing embeddings from: {faiss_index_path}")
        vectorstore = FAISS.load_local(onedrive_path, embeddings, allow_dangerous_deserialization=True)
    else:
        print(f"Embeddings do not exist. Generating new embeddings and saving to: {faiss_index_path}")
        vectorstore = FAISS.from_texts(source_text_chunks, embeddings)
        vectorstore.save_local(onedrive_path)

    return vectorstore

In [10]:
pdf_vectorstore = load_or_create_embeddings(chunks, ONEDRIVE_PATH_PDF)

Loading existing embeddings from: /Users/Shared/RAG-for-drug-pdf-files/database/index.faiss


Key Differences:
Direct Search (similarity_search) vs. Search Using Retriever (retriever.invoke())

similarity_search is a more direct method where the search is solely based on the similarity of text embeddings. You can control the number of results using the k parameter.
retriever.invoke() can apply more advanced query and result processing mechanisms (e.g., filtering, additional optimization steps). This may result in different, even subtly varying, outcomes.
Control Over the Number of Results:
In similarity_search, you can explicitly control the number of results returned by adjusting the k parameter.
In retriever.invoke(), the number of results depends on the retriever's implementation. You don’t have direct control over the number of results unless configured explicitly.
When to Use?
similarity_search:
Use it when you want a straightforward and quick way to find the most similar documents based on embeddings. It’s especially useful when you need full control over the number of results.

retriever.invoke():
Use it when you need a more flexible and advanced search mechanism. It’s suitable if you want to combine the results with other functionalities, such as applying additional filtering or processing that can influence the search or result handling.

In [11]:
# Test the functionality with a query
query = "What are the side effects of the drug Ifapidin?"

# Perform similarity search on the query
docs = pdf_vectorstore.similarity_search(query, k=3)

# Display the results
for i, doc in enumerate(docs):
    print(f"Result {i+1}: {doc.page_content}")

Result 1: krwiotwórczego.  
 
Decyzję o wznowieniu leczenia produktem Ifapidin należy podejmować na podstawie oceny objawów 
klinicznych i wyników badań laboratoryjnych.   
 
Reakcje krzyżowe między tienopirydynami  
Należy zebrać od pacjentów wywiad dotyczący nadwrażliwości na inną tienopirydynę (na przykład 
klopidogrel, prasugrel), ponieważ opisywano krzyżowe reakcje alergiczne między tienopirydynami 
(patrz punkt 4.8). Tienopirydyny mogą powodować łagodne do ciężkich reakcje alergiczne, takie jak 
wysypka, obrzęk naczynioruchowy lub hematologiczne reakcje krzyżowe, jak trombocytopenia i 
neutropenia. Pacjenci, u których wcześniej występowała reakcja alergiczna i (lub) hematologiczna na 
jakąś tienopirydynę, mogą być zagrożeni większym ryzykiem wystąpienia takiej samej lub innej 
 
 
 
 
4
reakcji na inny lek z grupy tienopirydyn. U pacjentów ze stwierdzoną alergią na tienopirydyny zaleca 
się obserwację w kierunku objawów nadwrażliwości.   
 
Hemostaza 
Produkt należy stosować ze s

In [12]:
# Test the functionality with a query
query = "What is the main chemical substance in Bisocard?"

# Perform similarity search on the query
docs = pdf_vectorstore.similarity_search(query, k=3)

# Display the results
for i, doc in enumerate(docs):
    print(f"Result {i+1}: {doc.page_content}")

Result 1: Data wydania pierwszego pozwolenia na dopuszczenie do obrotu: 29 kwietnia 2004 
Data ostatniego przedłużenia pozwolenia: 23 września 2013 
 
 
10. 
DATA ZATWIERDZENIA LUB CZĘŚCIOWEJ ZMIANY CHARAKTERYSTYKI 
PRODUKTU LECZNICZEGO 
 
27.07.2021 
 
1 
 
CHARAKTERYSTYKA PRODUKTU LECZNICZEGO 
 
 
1.  
NAZWA PRODUKTU LECZNICZEGO 
 
Bisocard, 5 mg, tabletki powlekane 
Bisocard, 10 mg, tabletki powlekane 
 
 
2.  
SKŁAD JAKOŚCIOWY I ILOŚCIOWY  
 
Bisocard, 5 mg, tabletki powlekane:  
Jedna tabletka powlekana zawiera 5 mg bisoprololu fumaranu (Bisoprololi fumaras). 
Substancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 120 mg. 
Bisocard, 10 mg, tabletki powlekane:  
Jedna tabletka powlekana zawiera 10 mg bisoprololu fumaranu (Bisoprololi fumaras).  
Substancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 115 mg. 
 
Pełny wykaz substancji pomocniczych, patrz punkt 6.1. 
 
 
3.  
POSTAĆ FARMACEUTYCZNA 
 
Tabletka powlekana. 
 
5 mg: Jasnożółte, okrągłe, obust

In [13]:
retriever = pdf_vectorstore.as_retriever()
retriever.invoke("What are the side effects of the drug Ifapidin?")
     

[Document(id='204416e2-6e77-4445-8e1c-33f9f347ab05', metadata={}, page_content='krwiotwórczego.  \n \nDecyzję o wznowieniu leczenia produktem Ifapidin należy podejmować na podstawie oceny objawów \nklinicznych i wyników badań laboratoryjnych.   \n \nReakcje krzyżowe między tienopirydynami  \nNależy zebrać od pacjentów wywiad dotyczący nadwrażliwości na inną tienopirydynę (na przykład \nklopidogrel, prasugrel), ponieważ opisywano krzyżowe reakcje alergiczne między tienopirydynami \n(patrz punkt 4.8). Tienopirydyny mogą powodować łagodne do ciężkich reakcje alergiczne, takie jak \nwysypka, obrzęk naczynioruchowy lub hematologiczne reakcje krzyżowe, jak trombocytopenia i \nneutropenia. Pacjenci, u których wcześniej występowała reakcja alergiczna i (lub) hematologiczna na \njakąś tienopirydynę, mogą być zagrożeni większym ryzykiem wystąpienia takiej samej lub innej \n \n \n \n \n4\nreakcji na inny lek z grupy tienopirydyn. U pacjentów ze stwierdzoną alergią na tienopirydyny zaleca \nsię ob

In [14]:
retriever.invoke("What is the main chemical substance in Bisocard?")

[Document(id='669ecbde-8201-4d4c-b9f1-92373cf77147', metadata={}, page_content='Data wydania pierwszego pozwolenia na dopuszczenie do obrotu: 29 kwietnia 2004 \nData ostatniego przedłużenia pozwolenia: 23 września 2013 \n \n \n10. \nDATA ZATWIERDZENIA LUB CZĘŚCIOWEJ ZMIANY CHARAKTERYSTYKI \nPRODUKTU LECZNICZEGO \n \n27.07.2021 \n \n1 \n \nCHARAKTERYSTYKA PRODUKTU LECZNICZEGO \n \n \n1.  \nNAZWA PRODUKTU LECZNICZEGO \n \nBisocard, 5 mg, tabletki powlekane \nBisocard, 10 mg, tabletki powlekane \n \n \n2.  \nSKŁAD JAKOŚCIOWY I ILOŚCIOWY  \n \nBisocard, 5 mg, tabletki powlekane:  \nJedna tabletka powlekana zawiera 5 mg bisoprololu fumaranu (Bisoprololi fumaras). \nSubstancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 120 mg. \nBisocard, 10 mg, tabletki powlekane:  \nJedna tabletka powlekana zawiera 10 mg bisoprololu fumaranu (Bisoprololi fumaras).  \nSubstancja pomocnicza o znanym działaniu: laktoza jednowodna w ilości 115 mg. \n \nPełny wykaz substancji pomocniczych, patrz

In [15]:
# Initialize GPT model for direct questions
response = model.invoke("Who is the president of Poland?")

# Display GPT response
print(response)

content='The current president of Poland is Andrzej Duda.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 14, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-12278abd-0c8f-460d-8a49-c6b2c0afa768-0' usage_metadata={'input_tokens': 14, 'output_tokens': 12, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [18]:
parser = StrOutputParser()

chain = model | parser 
print(chain.invoke("Is Poland i  UE?"))

Yes, Poland is a member of the European Union (EU). It joined the EU on May 1, 2004.


In [19]:
template = """
You are an assistant specializing in medical and pharmaceutical information, providing precise answers based on
the provided context. Your task is to retrieve information from the official drug leaflets (ChPLs).

- Answer the question in English based on the given context.
- Include only the relevant details from the context in your answer.
- If the context does not contain enough information to answer the question, respond with: "I don't know based on the provided information."
- Avoid including unrelated information.

Context: {context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)
print(prompt.format(context="Here is some context", question="Here is a question"))


You are an assistant specializing in medical and pharmaceutical information, providing precise answers based on
the provided context. Your task is to retrieve information from the official drug leaflets (ChPLs).

- Answer the question in English based on the given context.
- Include only the relevant details from the context in your answer.
- If the context does not contain enough information to answer the question, respond with: "I don't know based on the provided information."
- Avoid including unrelated information.

Context: Here is some context

Question: Here is a question



In [20]:
chain = prompt | model | parser

chain.invoke({
    "context": "Anna's sister is Susan", 
    "question": "Who is Susan's sister?"
})


"Susan's sister is Anna."

In [21]:
chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
    }
    | prompt
    | model
    | parser
)

In [22]:
# Define questions to test the chain
questions = [
    "what are the contraindications to using optilyte",
    "how to take singulair",
    "Does Bisocard lower blood pressure?",
    "Did you base your answers to the above questions only on the context?",
    "in what form does Aglan come?",
    "jaki skład ma Mizetam?",
    "z jakimi lekami reaguje stepcil?"
]

# Loop through questions and test the chain
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print("*************************\n")

Question: what are the contraindications to using optilyte
Answer: Based on the provided information, the contraindications to using Optilyte include:
- Hypersensitivity to the active substances or any of the excipients listed in section 6.1
- Acute renal failure
- Hypernatremia
- Hyperkalemia
- Hypercalcemia
- Hypermagnesemia
- Pulmonary edema

Therefore, the contraindications to using Optilyte are hypersensitivity to its components, acute renal failure, hypernatremia, hyperkalemia, hypercalcemia, hypermagnesemia, and pulmonary edema.
*************************

Question: how to take singulair
Answer: Singulair can be taken orally. Singulair Mini granules can be administered directly orally or mixed with soft food. The tablets should be swallowed whole. Singulair can be taken regardless of meals.
*************************

Question: Does Bisocard lower blood pressure?
Answer: Based on the provided information, Bisocard (bisoprolol) can lower blood pressure as it is mentioned that the s

In [28]:
# Define questions to test the chain
questions = [
    "can zofenil be divided into equal doses or crumbled",
    "do you know sorgifer durules?",
    "can acard cause stomach ulcers?",
    "What is the best drug for strong pain?",
    "what is nutriflex?",
    "Find me similar drugs by their indications to Ifapidin",
    "who should use losartan with caution??"
]

# Loop through questions and test the chain
for question in questions:
    print(f"Question: {question}")
    print(f"Answer: {chain.invoke({'question': question})}")
    print("*************************\n")

Question: can zofenil be divided into equal doses or crumbled
Answer: Zofenil 7,5 can be divided into equal doses, while Zofenil Plus cannot be divided into equal doses but can be crumbled for easier swallowing.
*************************

Question: do you know sorgifer durules?
Answer: Based on the provided information, Sorbifer Durules is a medication that should be taken orally by swallowing the tablets whole with water before or during a meal, depending on gastrointestinal tolerance. It is advised not to chew, suck, or hold the tablets in the mouth. Additionally, patients with swallowing difficulties should take Sorbifer Durules with caution to avoid mucosal inflammation or ulceration in the mouth. It is important to note that Sorbifer Durules contains iron sulfate, and accidental ingestion of these tablets can lead to serious respiratory complications.
*************************

Question: can acard cause stomach ulcers?
Answer: Yes, the medicinal product Acard Cor should be used wi